# IND320 project work – Part 4

This notebook documents the currency API, timestamp alignment, River reservoir forecast experiments, interactive-app design, and the required project log.

**GitHub repository:** https://github.com/M03ZGH/ind320-app  
**Streamlit app:** https://ind320-data-to-decision-moez.streamlit.app/

Run from the `ind320-app` folder with the `ind320` Python environment. The NVE, Norges Bank and Open-Meteo APIs are public. The Streamlit app downloads weather directly from Open-Meteo (`src/ind320_app/weather.py`); only the ENTSO-E cross-border flows are read from MongoDB and need a `[mongo].uri` secret.

Monthly resolution is used for model inputs and correlations: it retains a useful history while keeping a full set of model comparisons quick to run.

## App structure sketch

The page order follows a simple path from orientation, through exploration, to prediction and source-specific details:

```mermaid
flowchart TD
    A[Overview] --> B[Explore]
    B --> C[NVE data and reservoir plots]
    B --> D[Weather-region map]
    B --> E[Weather / filling correlation]
    B --> F[Weather anomalies and time series]
    A --> G[Predict]
    G --> H[River monthly filling forecast]
    A --> I[Data sources]
    I --> J[Norges Bank currency]
    I --> K[ENTSO-E flows]
```

The app has a grouped navigation bar (Project, Explore, Predict, Data sources). The map weather property, the map period/week and the forecast horizon are kept in Streamlit session state, and a click on an area stores the selected price area, which becomes the default area on the correlation, weather-outlier and forecast pages. The map can draw price areas (EL, NO1-NO5), Norway as a whole (NO, the union of the price areas) and the NVE precipitation regions, coloured by a weather property or by the reservoir filling for a chosen week. VASS (water-area) polygons are not available in the project, so VASS is explored in the data and plot pages but not drawn on the map. The old LineChartColumn page was merged into the NVE Data page.

## 1. Setup and shared functions

The notebook imports the same functions used by Streamlit. The working-directory check makes the project-relative data paths reproducible.

In [12]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display

In [13]:
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in (Path.cwd(), Path.cwd().parent)
     if (candidate / "src" / "ind320_app").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the ind320-app folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


print("Project:", PROJECT_ROOT)
print("Weather-region shapefile present:", (PROJECT_ROOT / "data" / "NVE_Region.shp").exists())
print("Bidding-area (EL) shapefile present:", (PROJECT_ROOT / "data" / "Budområder.shp").exists())

Project: /Users/moezyadgar/GitHub/IND320/ind320-app
Weather-region shapefile present: True
Bidding-area (EL) shapefile present: True


## 1b. Function library for Part 4

The functions for this part (currency downloader, sliding-window correlation, River forecast and the Plotly helpers) live in `src/ind320_app/part4.py`. The Streamlit pages import the same module, so the notebook and the app run exactly the same code.

In [14]:
# Shared Part 3 helpers (weather download, point sampling) and NVE data loader.
from ind320_app.data import load_reservoir_data
from ind320_app.part3 import download_weather, sample_region_coordinates

# Part 4 functions, imported from src/ind320_app/part4.py
from ind320_app.part4 import (
    download_currency_rates,
    forecast_reservoir_filling,
    sliding_window_correlation,
)

## 2. Euro and US dollar exchange rates

Norges Bank's SDMX API series identifiers are `B.EUR.NOK.SP` and `B.USD.NOK.SP`. The helper downloads semicolon-separated CSV and returns a date-indexed frame in NOK per unit of foreign currency. The sample below uses dates with known published observations.

In [15]:
# Request actual rates from Norges Bank; surface a network issue if the runtime blocks this host.
try:
    currency_rates = download_currency_rates("2024-01-01", "2024-01-05")
except requests.exceptions.RequestException as error:
    print("Live Norges Bank request was unavailable in this runtime:", error)
else:
    display(currency_rates)
    assert {"EUR_NOK", "USD_NOK"}.issubset(currency_rates.columns)
    assert not currency_rates.empty

,EUR_NOK,USD_NOK
date,,
2024-01-02,11.2815,10.2971
2024-01-03,11.3200,10.3672
2024-01-04,11.2845,10.3027
2024-01-05,11.3090,10.3553


In [16]:
# Parse a short CSV excerpt retrieved from the official API to demonstrate the downloader's parsing path offline.
class _RecordedNorgesBankResponse:
    text = (
        "TIME_PERIOD;OBS_VALUE\n"
        "2024-01-02;11.2815\n"
        "2024-01-03;11.3200\n"
    )

    def raise_for_status(self):
        return None


class _RecordedNorgesBankSession:
    def get(self, url, params, timeout):
        return _RecordedNorgesBankResponse()


recorded_rates = download_currency_rates(
    "2024-01-01", "2024-01-05", currencies=("EUR",), session=_RecordedNorgesBankSession()
)
display(recorded_rates)
assert recorded_rates.loc[pd.Timestamp("2024-01-02"), "EUR_NOK"] == 11.2815
print("Verified public endpoint:", "https://data.norges-bank.no/api/data/EXR/B.EUR.NOK.SP")

,EUR_NOK
date,
2024-01-02,11.2815
2024-01-03,11.3200


Verified public endpoint: https://data.norges-bank.no/api/data/EXR/B.EUR.NOK.SP


## 3. Timezone investigation and alignment

The NVE payload uses a weekly calendar date (`dato_Id`) without a clock time or timezone. ENTSO-E observations in this project are parsed as timezone-aware UTC instants. Open-Meteo is requested with `timezone="UTC"`. Norges Bank's daily spot series has a date label, and the API metadata identifies the observation as the ECB concertation-time rate around 14:15 CET; it is not a UTC-midnight instant.

The practical solution is to preserve date-only NVE and Norges Bank observations as calendar dates, keep hourly instants in UTC while joining, and convert timestamped weather/flow observations to `Europe/Oslo` only when a local daily interpretation is intended. Monthly resampling is used for the forecast/correlation inputs, avoiding false hour-level matches and DST shifts.

In [17]:
timezone_findings = pd.DataFrame([
    {"Source": "Magasinstatistikk / NVE", "Returned time": "Weekly date only", "Handling": "Keep the Norwegian calendar date; no timezone conversion."},
    {"Source": "ENTSO-E", "Returned time": "UTC timestamp", "Handling": "Keep timezone-aware UTC instants; convert only for local display/aggregation."},
    {"Source": "Open-Meteo", "Returned time": "Hourly; requested with timezone=UTC", "Handling": "Keep UTC for joins; convert to Europe/Oslo for local-day analysis."},
    {"Source": "Norges Bank", "Returned time": "Business-date label; ECB observation around 14:15 CET", "Handling": "Keep as a date; do not localize to UTC midnight."},
])
display(timezone_findings)

,Source,Returned time,Handling
0,Magasinstatistikk / NVE,Weekly date only,Keep the Norwegian calendar date; no timezone ...
1,ENTSO-E,UTC timestamp,Keep timezone-aware UTC instants; convert only...
2,Open-Meteo,Hourly; requested with timezone=UTC,Keep UTC for joins; convert to Europe/Oslo for...
3,Norges Bank,Business-date label; ECB observation around 14...,Keep as a date; do not localize to UTC midnight.


## 4. Convert the shapefiles to GeoJSON

Two NVE shapefiles are reprojected from EPSG:25833 to WGS84 for web maps, simplified slightly, and written to the app's `data/` folder:

* `Budområder.shp` – the electricity bidding (price) areas NO1–NO5. **These are the polygons used for the electricity areas**, including NO3.
* `NVE_Region.shp` – the NVE precipitation/weather regions (RV, RM, RN, RØ, RS), kept as a separate map layer.

Part 3 mapped NO3 to the precipitation region RM. Peer feedback pointed out that NO3 is a price area and must use the NO3 polygon of the electricity-area layer; RM overlaps only about half of NO3 (see the check below). The NVE area type `NO` (Norway as a whole) is drawn as the union of the five price areas (`nve_norway.geojson`). No VASS (water-area) polygon file is available in this project, so VASS can be explored in the data and plot pages but not drawn on the map.

In [18]:
# Convert both NVE shapefiles to web-map GeoJSON (WGS84).
el_shapefile = PROJECT_ROOT / "data" / "Budområder.shp"
weather_shapefile = PROJECT_ROOT / "data" / "NVE_Region.shp"
for path in (el_shapefile, weather_shapefile):
    if not path.exists():
        raise FileNotFoundError(f"Place {path.name} and its companion files in data/.")

# --- Electricity bidding areas (price areas NO1-NO5) ---
el_areas = gpd.read_file(el_shapefile).to_crs("EPSG:4326")
el_areas["EL"] = el_areas["budomr"].str.replace(" ", "", regex=False)  # "NO 3" -> "NO3"
el_areas["omrnr"] = el_areas["EL"].str[-1].astype(int)  # joins to omrnr in the NVE data
el_areas["geometry"] = el_areas.geometry.simplify(0.001, preserve_topology=True)
el_geojson_path = PROJECT_ROOT / "data" / "nve_el_areas.geojson"
el_geojson_path.write_text(
    el_areas[["EL", "omrnr", "geometry"]].to_json(drop_id=True), encoding="utf-8"
)
print("EL GeoJSON:", el_geojson_path, "| bytes:", el_geojson_path.stat().st_size)
display(el_areas[["EL", "omrnr"]])

# --- Norway as a whole (area type "NO", omrnr 0): the union of the price areas ---
norway = gpd.GeoDataFrame(
    {"NO": ["NO"], "omrnr": [0]}, geometry=[el_areas.geometry.union_all()], crs=el_areas.crs
)
(PROJECT_ROOT / "data" / "nve_norway.geojson").write_text(
    norway.to_json(drop_id=True), encoding="utf-8"
)

# --- NVE precipitation / weather regions ---
regions = gpd.read_file(weather_shapefile).to_crs("EPSG:4326")

# The source DBF contains UTF-8 bytes interpreted as Latin-1 on some systems.
def repair_dbf_text(value):
    if not isinstance(value, str):
        return value
    try:
        return value.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value

for column in ("REGION", "REGIONNAVN"):
    regions[column] = regions[column].map(repair_dbf_text)
regions["geometry"] = regions.geometry.simplify(0.001, preserve_topology=True)
geojson_path = PROJECT_ROOT / "data" / "nve_weather_regions.geojson"
geojson_path.write_text(
    regions[["REGION", "REGIONNAVN", "geometry"]].to_json(drop_id=True),
    encoding="utf-8",
)
print("Weather-region GeoJSON:", geojson_path, "| bytes:", geojson_path.stat().st_size)

# Why RM was wrong for NO3: share of the NO3 polygon covered by each weather region.
no3_polygon = el_areas.loc[el_areas["EL"] == "NO3", "geometry"].iloc[0]
no3_overlap = pd.Series({
    row.REGION: row.geometry.intersection(no3_polygon).area / no3_polygon.area
    for row in regions.itertuples()
}, name="share_of_NO3").round(2)
display(no3_overlap.to_frame())

EL GeoJSON: /Users/moezyadgar/GitHub/IND320/ind320-app/data/nve_el_areas.geojson | bytes: 136273


,EL,omrnr
0,NO1,1
1,NO2,2
2,NO3,3
3,NO4,4
4,NO5,5


Weather-region GeoJSON: /Users/moezyadgar/GitHub/IND320/ind320-app/data/nve_weather_regions.geojson | bytes: 164671


,share_of_NO3
RV,0.39
RM,0.51
RN,0.00
RØ,0.11
RS,0.00


## 5. River monthly reservoir forecast

The selected target is EL area 1, which has a long weekly history. It is averaged to month-start values before fitting. A River online linear regressor receives lagged filling, cyclical month features, and (optionally) selected exogenous columns. The benchmark is chronological one-step validation against persistence. We compare history lengths, lag counts, and learning rates rather than assuming a setting is best.

In [19]:
# Download the public NVE history and select one region for a reproducible experiment.
reservoirs = load_reservoir_data()
filling = (
    reservoirs[(reservoirs["area_type"] == "EL") & (reservoirs["area_number"] == 1)]
    .set_index("date")["filling_degree"]
    .dropna()
)
monthly_filling = filling.resample("MS").mean().dropna()
print("NVE rows:", len(reservoirs), "| EL1 monthly observations:", len(monthly_filling))
display(monthly_filling.tail())

NVE rows: 14976 | EL1 monthly observations: 383


date
2026-07-01    0.716663
2026-08-01    0.713820
2026-09-01    0.735844
2026-10-01    0.756352
2026-11-01    0.774827
Freq: MS, Name: filling_degree, dtype: float64

In [20]:
# Test several history/lag/learning-rate settings; omit combinations with too few training months.
settings = []
for history_months in (24, 60, 120):
    for lag_count in (3, 6, 12):
        if lag_count >= history_months - 12:
            continue
        for learning_rate in (0.001, 0.01, 0.05):
            result = forecast_reservoir_filling(
                filling,
                history_months=history_months,
                horizon=6,
                lags=lag_count,
                learning_rate=learning_rate,
            )
            settings.append({
                "history_months": history_months,
                "lags": lag_count,
                "learning_rate": learning_rate,
                "River_MAE": result.mae,
                "persistence_MAE": result.persistence_mae,
            })

settings_results = pd.DataFrame(settings).sort_values("River_MAE").reset_index(drop=True)
display(settings_results.head(10).round(4))
best_setting = settings_results.iloc[0]
print("Best tested setting:", best_setting.to_dict())

,history_months,lags,learning_rate,River_MAE,persistence_MAE
0,120,12,0.01,0.0601,0.1085
1,120,3,0.01,0.0726,0.1085
2,120,6,0.01,0.0739,0.1085
3,60,6,0.05,0.0770,0.1085
4,120,6,0.05,0.0803,0.1085
5,120,3,0.05,0.0838,0.1085
6,60,3,0.05,0.0848,0.1085
7,24,3,0.05,0.1037,0.0729
8,120,12,0.05,0.1516,0.1085
9,60,3,0.01,0.2259,0.1085


Best tested setting: {'history_months': 120.0, 'lags': 12.0, 'learning_rate': 0.01, 'River_MAE': 0.06010783505325654, 'persistence_MAE': 0.10847375583333334}


In [21]:
# Train the default model using the best setting from the test grid.
default_result = forecast_reservoir_filling(
    filling,
    history_months=120,
    horizon=6,
    lags=12,
    learning_rate=0.01,
)
forecast_figure = go.Figure()
forecast_figure.add_trace(go.Scatter(
    x=default_result.history.index,
    y=default_result.history,
    name="Observed monthly filling",
    mode="lines",
))
forecast_figure.add_trace(go.Scatter(
    x=default_result.forecast.index,
    y=default_result.forecast,
    name="River forecast",
    mode="lines+markers",
))
forecast_figure.update_layout(
    title="EL1 monthly filling: River forecast",
    xaxis_title="Month",
    yaxis_title="Filling fraction",
    yaxis={"range": [0, 1]},
    template="plotly_white",
)
display(forecast_figure)
print({
    "walk_forward_MAE": round(default_result.mae, 4),
    "persistence_MAE": round(default_result.persistence_mae, 4),
    "forecast": default_result.forecast.round(3).to_dict(),
})

{'walk_forward_MAE': 0.0601, 'persistence_MAE': 0.1085, 'forecast': {Timestamp('2026-12-01 00:00:00'): 0.631, Timestamp('2027-01-01 00:00:00'): 0.474, Timestamp('2027-02-01 00:00:00'): 0.316, Timestamp('2027-03-01 00:00:00'): 0.203, Timestamp('2027-04-01 00:00:00'): 0.211, Timestamp('2027-05-01 00:00:00'): 0.361}}


## 6. Sliding weather/filling correlations

The correlation page exposes weather property, lead lag, and window length. To explore those controls with actual data, the next cell samples 100 locations inside the **NO3 bidding-area polygon** (`Budområder.shp`), downloads hourly ERA5 observations from Open-Meteo (cached in `data/weather_cache_el_<points>/`), aggregates them to monthly values, and compares them with EL3 monthly filling (same price area). Positive lag means weather leads filling. With only 12 months in the 2024 weather sample, results are exploratory and especially sensitive to the six-month window.

**Reduction note:** the assignment asks for 1000 locations per area. Open-Meteo API restrictions make that many requests hard to complete, so this notebook uses 100 random points per area (a pragmatic reduction approved by the lecturer). Averaged over a month, 100 random points in one polygon should give almost the same value as 1000.

In [22]:
# Sample reproducible points inside each bidding-area polygon (keys "NO1".."NO5").
# REDUCTION: the assignment asks for 1000 locations per area, but Open-Meteo's API
# limits make that many requests unreliable. 100 random points per area are used
# instead (the lecturer allowed a pragmatic reduction). Monthly means over 100
# random points inside one polygon are very close to those over 1000, and the
# downloads are cached in data/weather_cache_el_<points>/.
POINTS_PER_AREA = 100
# A separate cache folder (named after the number of points) keeps these batches apart from the Part 3 cache and from other sample sizes.
coordinates = sample_region_coordinates(el_shapefile, points_per_region=POINTS_PER_AREA, random_state=42)
weather_2024 = download_weather(
    "NO3",
    "2024-01-01",
    "2024-12-31",
    coordinates,
    batch_size=10,
    request_pause=0,
    cache_dir=PROJECT_ROOT / "data" / f"weather_cache_el_{POINTS_PER_AREA}",
    show_progress=False,
)

# Weather is NO3, so compare against the filling of the same price area (EL3).
filling_el3 = (
    reservoirs[(reservoirs["area_type"] == "EL") & (reservoirs["area_number"] == 3)]
    .set_index("date")["filling_degree"]
    .dropna()
)
correlation_checks = []
for property_name in ("temperature_2m", "precipitation", "wind_speed_10m"):
    for lag in (0, 1, 3):
        correlation = sliding_window_correlation(
            weather_2024[property_name], filling_el3, lag=lag, window=6
        )
        correlation_checks.append({
            "property": property_name,
            "weather_lead_months": lag,
            "rolling_windows": len(correlation),
            "mean_r": correlation["correlation"].mean(),
            "minimum_r": correlation["correlation"].min(),
            "maximum_r": correlation["correlation"].max(),
        })
correlation_results = pd.DataFrame(correlation_checks)
display(correlation_results.round(3))

,property,weather_lead_months,rolling_windows,mean_r,minimum_r,maximum_r
0,temperature_2m,0,8,0.052,-0.871,0.856
1,temperature_2m,1,7,0.345,-0.579,0.982
2,temperature_2m,3,5,0.810,0.591,0.913
3,precipitation,0,8,0.687,0.451,0.868
4,precipitation,1,7,0.528,0.041,0.889
5,precipitation,3,5,0.284,-0.566,0.863
6,wind_speed_10m,0,8,0.374,-0.191,0.747
7,wind_speed_10m,1,7,0.047,-0.590,0.713
8,wind_speed_10m,3,5,-0.242,-0.887,0.248


## 7. Project log (300–500 words)

I added a reusable downloader for Norges Bank's daily EUR/NOK and USD/NOK spot rates. It reads the public SDMX API as CSV and returns a date-indexed frame. The session is injectable, so parsing can be tested offline.

I compared timestamp conventions. Magasinstatistikk gives weekly calendar dates without a timezone; ENTSO-E data is timezone-aware UTC; Open-Meteo was requested in UTC; Norges Bank labels daily rates by business date (the spot fixing is around 14:15 CET). NVE and Norges Bank dates are kept as dates, instant observations stay timezone-aware, and UTC is converted to Europe/Oslo only for local-day analysis. Monthly aggregation avoids false hour-level alignment and DST shifts.

The forecast function resamples one NVE region to monthly values and trains River's online linear regression on filling lags plus cyclical month features. History, horizon, lags, learning rate and exogenous sources are inputs. Monthly resolution keeps a full grid of settings fast. I tested 24-, 60- and 120-month histories, 3/6/12 lags and learning rates 0.001/0.01/0.05 with walk-forward validation. 120 months, 12 lags and 0.01 gave the lowest MAE (about 0.060 against about 0.109 for persistence) and is the Streamlit default. Exogenous inputs are lagged one month, and the latest value is held constant over the horizon because future weather is unknown.

I explored sliding correlations between weather and EL3 filling. Temperature correlates positively, and the correlation is clearly higher when weather leads filling by one month than at lag 0. Precipitation and wind change sign with lag and window length. The weather sample covers only 2024, so the six-month windows rest on few points and these patterns are hypotheses, not evidence of predictive skill.

Weather points: the assignment asks for 1000 locations per area. Open-Meteo's API limits made this unreliable, so I used 100 random points per bidding area and cache the downloads. Monthly means over random points change little with more points, but the choice is a reduction.

For the app I grouped pages into Explore, Predict and Data sources, replaced static charts with Plotly, and built a map with GeoJSON price areas, a Norway layer and precipitation regions. The map is coloured by a weather property or reservoir filling for a chosen week, and a click outlines and stores the selected area. NO3 now uses the NO3 price-area polygon, not the RM weather region as in Part 3, following peer feedback. VASS polygons are not available. In the app, weather is downloaded directly from Open-Meteo per price area (5 random points per area, one request per calendar year, cached for 24 hours), so no database sits between the app and the weather API. The bonus tasks I chose were waiting time (a progress bar for downloads, spinners and caching of data loads, the River forecast, STL and the spectrogram) and error handling (readable messages when an API, the database or the data fail, including Open-Meteo's rate limit).

## 8. AI usage

AI tools were used to clarify the Norges Bank API response, River's online-learning interface, timezone alignment, and Streamlit navigation/Plotly behavior. Suggestions were checked against the live API data, installed River signatures, and real NVE/weather observations; I reviewed and adapted the implementation rather than accepting generated code without validation.